In [ ]:
import os
import sys
from rdkit import Chem
from rdkit.Chem import QED, RDConfig, Crippen, Lipinski, Descriptors
import numpy as np
from easydict import EasyDict
import pickle
import pandas as pd

sys.path.append(os.path.join(RDConfig.RDContribDir, "SA_Score"))
import sascorer as sa


def get_sdf(sdf_file):
    supp = Chem.SDMolSupplier(sdf_file, sanitize=False)
    mols = [i for i in supp][0]
    return mols


def obey_lipinski(mol):
    Chem.SanitizeMol(mol)
    rule_1 = Descriptors.ExactMolWt(mol) < 500
    rule_2 = Lipinski.NumHDonors(mol) <= 5
    rule_3 = Lipinski.NumHAcceptors(mol) <= 10
    rule_4 = (logp := Crippen.MolLogP(mol) >= -2) & (logp <= 5)
    rule_5 = Lipinski.NumRotatableBonds(mol) <= 10
    return np.sum([int(a) for a in [rule_1, rule_2, rule_3, rule_4, rule_5]])


def write_pkl(list, filename):
    with open(filename, "wb") as f:
        pickle.dump(list, f)


def standrize(mol):
    return Chem.MolFromSmiles(Chem.MolToSmiles(mol))


def prop(mol):
    mol_dict = [
        QED.qed(mol),
        sa.calculateScore(mol),
        Descriptors.MolWt(mol),
        Crippen.MolLogP(mol),
        Descriptors.TPSA(mol),
        obey_lipinski(mol),
    ]
    return mol_dict


def prop_smi(smi):
    mol = Chem.MolFromSmiles(smi)
    if mol:
        mol_dict = [
            QED.qed(mol),
            sa.calculateScore(mol),
            Descriptors.MolWt(mol),
            Crippen.MolLogP(mol),
            Descriptors.TPSA(mol),
            obey_lipinski(mol),
        ]
        return mol_dict


def read_pkl(pklfile):
    with open(pklfile, "rb") as f:
        a = pickle.load(f)
    return a

/home/sorui/anaconda3/envs/AMP/lib/python3.8/site-packages/tqdm/auto.py:22: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:
li = read_pkl("./test_result/rmse_40s_results.pkl")
ova = []
for pok in li:
    data = [
        [
            sin["affinity"],
            sin["qed"],
            sin["SAS"],
            sin["MW"],
            sin["LogP"],
            sin["TPSA"],
            sin["Lp5"],
        ]
        for sin in pok
        if sin
    ]
    ova.append(np.mean(data, axis=0))

import pandas as pd


def stru_res(itr):
    mol_dict = EasyDict(
        {
            "Vina Score": "%.3f" % itr[0],
            "qed": "%.3f" % itr[1],
            "SAS": "%.3f" % itr[2],
            "MW": "%.3f" % itr[3],
            "LogP": "%.3f" % itr[4],
            "TPSA": "%.3f" % itr[5],
            "Lp5": "%.3f" % itr[6],
        }
    )
    return pd.DataFrame(mol_dict, index=[0]).T


means = np.mean(ova, axis=0)
medians = np.median(ova, axis=0)

In [3]:
stru_res(means)

,0
Vina Score,-6.862
qed,0.568
SAS,4.166
MW,355.757
LogP,2.319
TPSA,87.320
Lp5,4.748


In [4]:
stru_res(medians)

,0
Vina Score,-7.077
qed,0.568
SAS,4.232
MW,367.252
LogP,2.363
TPSA,87.660
Lp5,4.763
